# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamKapoor22/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Model Choice:** Random Forest Classifier & Logistic Regression.

**Justification:** Random Forest handles non-linear relationships between staleness and search volume effectively without heavy feature scaling, while Logistic Regression serves as a simple probabilistic baseline. They allow clear feature importance interpretability to beat our Week-4 rule-based baseline.

In [6]:
import os, sys
import pandas as pd
import numpy as np

# Load data safely from local folder or GitHub raw link
data_path = "data/raw/content_refresh_anonymized.csv"
raw_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(raw_url)

print("Data loaded successfully! Total rows:", len(df))

Data loaded successfully! Total rows: 30000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split Design:** 80/20 Stratified Train-Test Split based on trend_direction target (is_declining). This split ensures honest validation without leakage, keeping class distributions balanced across train and test sets.

In [7]:
from sklearn.model_selection import train_test_split

# Prepare Features & Target
feature_cols = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'word_count']
X = df[feature_cols].fillna(0)
y = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train set size: {len(X_train)} | Test set size: {len(X_test)}")
print(f"Target distribution in test set:\n{y_test.value_counts(normalize=True)}")

Train set size: 24000 | Test set size: 6000
Target distribution in test set:
trend_direction
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Week 4 Baseline Rules on Test Set
max_age = df['days_since_last_update'].max()
baseline_scores = (X_test['days_since_last_update'] / max_age) * 0.7 + (X_test['avg_position'] / 100) * 0.3
baseline_preds = (baseline_scores > 0.5).astype(int)

# 2. Train Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# Metrics Comparison Table
def compute_metrics(y_true, y_pred, y_prob=None):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC AUC': roc_auc_score(y_true, y_prob) if y_prob is not None else roc_auc_score(y_true, y_pred)
    }

metrics_df = pd.DataFrame({
    'W04 Baseline Rule': compute_metrics(y_test, baseline_preds, baseline_scores),
    'Random Forest Model': compute_metrics(y_test, rf_preds, rf_probs)
}).T

print("=== Model vs Baseline Performance Comparison ===")
print(metrics_df.round(4))

# Save metrics receipt JSON
os.makedirs("work/outputs", exist_ok=True)
metrics_df.to_json("work/outputs/w05_model_metrics.json", indent=4)
print("\n[SUCCESS] Metrics JSON generated at work/outputs/w05_model_metrics.json")

=== Model vs Baseline Performance Comparison ===
                     Accuracy  Precision  Recall  F1 Score  ROC AUC
W04 Baseline Rule      0.4575     0.3333  0.0009    0.0018   0.5496
Random Forest Model    0.6593     0.6494  0.8072    0.7198   0.7211

[SUCCESS] Metrics JSON generated at work/outputs/w05_model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**False Positives:** High staleness pages (days_since_last_update > 200) that remain stable due to strong domain authority or evergreen SERP intent.

**False Negatives:** Newer pages (content_age_days < 90) experiencing sharp rank drops due to competitor shifts, where staleness features didn't trigger early enough.

In [9]:
from sklearn.inspection import permutation_importance

# Permutation Importance
result = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)
perm_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance_Mean': result.importances_mean
}).sort_values(by='Importance_Mean', ascending=False)

print("=== Permutation Feature Importance ===")
print(perm_df.to_string(index=False))

# False Positives / Negatives Count
test_analysis = X_test.copy()
test_analysis['y_true'] = y_test
test_analysis['pred'] = rf_preds

fp = len(test_analysis[(test_analysis['y_true'] == 0) & (test_analysis['pred'] == 1)])
fn = len(test_analysis[(test_analysis['y_true'] == 1) & (test_analysis['pred'] == 0)])

print(f"\nFalse Positives Count: {fp}")
print(f"False Negatives Count: {fn}")

=== Permutation Feature Importance ===
               Feature  Importance_Mean
       impressions_90d         0.067750
      content_age_days         0.032767
            word_count         0.018467
          avg_position         0.017350
days_since_last_update         0.002150

False Positives Count: 1417
False Negatives Count: 627


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.